# PhoBERT trên ViCTSD
Huấn luyện & So sánh trực tiếp giữa Nhãn Gốc và Nhãn AI Re-annotated.

In [ ]:
# 1. Tự động chuẩn bị môi trường & Dữ liệu trên Kaggle / Colab
import os
import sys

if not any(os.path.exists(p) for p in ['data', '../data', 'multi-agent-annotation/data', '/kaggle/working/data']):
    print("Đang tự động clone repository và dữ liệu từ GitHub...")
    !git clone https://github.com/lechihoang/multi-agent-annotation.git
    if os.path.exists('multi-agent-annotation'):
        %cd multi-agent-annotation/notebooks

print("Môi trường đã sẵn sàng!")
!pip install -q transformers datasets torch accelerate seaborn matplotlib scikit-learn pandas

In [ ]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

# Tìm đường dẫn thư mục data
data_dirs = ['data', '../data', './data', 'multi-agent-annotation/data', '/kaggle/working/multi-agent-annotation/data', '/kaggle/working/data']
data_dir = None
for d in data_dirs:
    if os.path.exists(os.path.join(d, 'ViCTSD_train.csv')):
        data_dir = d
        break

if data_dir is None:
    raise FileNotFoundError("Không tìm thấy thư mục data chứa các file ViCTSD_*.csv!")

print(f"Thư mục dữ liệu: {data_dir}")

train_orig = pd.read_csv(os.path.join(data_dir, 'ViCTSD_train.csv')).fillna('')
valid_orig = pd.read_csv(os.path.join(data_dir, 'ViCTSD_valid.csv')).fillna('')
test_df = pd.read_csv(os.path.join(data_dir, 'ViCTSD_test.csv')).fillna('')

# Tải file nhãn do AI gán lại (re-annotated)
reann_train_path = os.path.join(data_dir, 'ViCTSD_train_reannotated.csv')
if not os.path.exists(reann_train_path):
    reann_train_path = os.path.join(data_dir, 'ViCTSD_train_labeled.csv')

reann_valid_path = os.path.join(data_dir, 'ViCTSD_valid_reannotated.csv')
if not os.path.exists(reann_valid_path):
    reann_valid_path = os.path.join(data_dir, 'ViCTSD_valid_labeled.csv')

train_new = pd.read_csv(reann_train_path).fillna('')
valid_new = pd.read_csv(reann_valid_path).fillna('')

train_new = train_new[train_new['Constructiveness'] != -1]
valid_new = valid_new[valid_new['Constructiveness'] != -1]

print(f"Train original: {len(train_orig)} mẫu")
print(f"Train AI re-annotated: {len(train_new)} mẫu")
print(f"Test dataset: {len(test_df)} mẫu")

In [ ]:
def evaluate_model(model_name, y_true, y_pred):
    print(f"--- {model_name} ---")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 Macro: {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(classification_report(y_true, y_pred))
    print("="*50)

def compare_and_plot_results(model_name, y_true, y_pred_orig, y_pred_new):
    acc_orig = accuracy_score(y_true, y_pred_orig)
    f1_orig = f1_score(y_true, y_pred_orig, average='macro')
    prec_orig = precision_score(y_true, y_pred_orig, average='macro')
    rec_orig = recall_score(y_true, y_pred_orig, average='macro')
    
    acc_new = accuracy_score(y_true, y_pred_new)
    f1_new = f1_score(y_true, y_pred_new, average='macro')
    prec_new = precision_score(y_true, y_pred_new, average='macro')
    rec_new = recall_score(y_true, y_pred_new, average='macro')
    
    df_res = pd.DataFrame({
        'Metric': ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'],
        'Original Labels': [acc_orig, f1_orig, prec_orig, rec_orig],
        'AI Re-annotated Labels': [acc_new, f1_new, prec_new, rec_new],
        'Delta (Diff)': [acc_new - acc_orig, f1_new - f1_orig, prec_new - prec_orig, rec_new - rec_orig]
    })
    
    print(f"\n=================== BẢNG SO SÁNH TRỰC TIẾP: {model_name} ===================")
    print(df_res.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("="*75)
    
    # Biểu đồ so sánh trực quan
    metrics_df = pd.melt(df_res, id_vars=['Metric'], value_vars=['Original Labels', 'AI Re-annotated Labels'],
                         var_name='Dataset', value_name='Score')
    
    plt.figure(figsize=(9, 5))
    ax = sns.barplot(data=metrics_df, x='Metric', y='Score', hue='Dataset', palette=['#3498db', '#e74c3c'])
    plt.title(f"So sánh hiệu năng: {model_name} (Original vs AI Re-annotated)", fontsize=13, fontweight='bold')
    plt.ylim(0, 1.08)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:.4f}',
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', fontsize=10, xytext=(0, 3),
                        textcoords='offset points')
                        
    plt.tight_layout()
    plt.show()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}")

y_test = test_df['Constructiveness'].astype(int).values

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def get_hf_dataset(df):
    data_dict = {
        'text': df['Comment'].tolist(),
        'label': df['Constructiveness'].astype(int).tolist()
    }
    return Dataset.from_dict(data_dict)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset_orig = get_hf_dataset(train_orig).map(tokenize_function, batched=True)
train_dataset_new = get_hf_dataset(train_new).map(tokenize_function, batched=True)
test_dataset = get_hf_dataset(test_df).map(tokenize_function, batched=True)

def train_and_eval(train_ds, test_ds, model_name):
    model = AutoModelForSequenceClassification.from_pretrained("vinai/phobert-base-v2", num_labels=2)
    
    try:
        training_args = TrainingArguments(
            output_dir=f"./phobert_results_{model_name.replace(' ', '_')}",
            eval_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=3,
            fp16=torch.cuda.is_available(),
            logging_steps=50,
            report_to="none"
        )
    except Exception:
        training_args = TrainingArguments(
            output_dir=f"./phobert_results_{model_name.replace(' ', '_')}",
            evaluation_strategy="epoch",
            save_strategy="epoch",
            learning_rate=2e-5,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=32,
            num_train_epochs=3,
            fp16=torch.cuda.is_available(),
            logging_steps=50,
            report_to="none"
        )
        
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds
    )
    
    print(f"Đang huấn luyện {model_name}...")
    trainer.train()
    
    preds_output = trainer.predict(test_ds)
    preds = np.argmax(preds_output.predictions, axis=-1)
    evaluate_model(model_name, y_test, preds)
    return preds

In [ ]:
print("=== 1. Training PhoBERT on Original Labels ===")
y_pred_phobert_orig = train_and_eval(train_dataset_orig, test_dataset, 'PhoBERT (Original Labels)')

In [ ]:
print("=== 2. Training PhoBERT on AI Re-annotated Labels ===")
y_pred_phobert_new = train_and_eval(train_dataset_new, test_dataset, 'PhoBERT (AI Re-annotated Labels)')

In [ ]:
# 3. SO SÁNH TRỰC TIẾP KẾT QUẢ & VẼ BIỂU ĐỒ
compare_and_plot_results("PhoBERT", y_test, y_pred_phobert_orig, y_pred_phobert_new)